<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">بستهٔ یکسان، ورودی محدود، اجرای محلی</h1>
<p style="text-align:right">درس 91 از 92 · مدل را از دفتر آزمایش بیرون بیاوریم، با چه قراردادی؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">83-deployment</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-15/chapter-03/83-deployment.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">قرارداد بودجهٔ درخواست و سازگاری <bdi dir="ltr">Vocabulary</bdi> را پیش از تولید بررسی کنید.</p><p style="text-align:right">پیش‌نیاز: <bdi dir="ltr">Context Window</bdi>، ذخیرهٔ وزن و فرق <bdi dir="ltr">Checkpoint</bdi> آموزش با بستهٔ <bdi dir="ltr">Inference</bdi>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۵۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر دو <bdi dir="ltr">Vocabulary</bdi> هم‌اندازه‌اند ولی ترتیبشان فرق دارد، آیا برابری <bdi dir="ltr">Shape</bdi> جدول‌ها کافی است؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import tempfile
from pathlib import Path
import torch
from mini_gpt.instruction import (run_tiny_experiment,format_prompt,TRAIN_EXAMPLES,
    save_instruction_bundle,load_instruction_bundle)
from mini_gpt.performance import InferenceSession,RequestLimits

torch.set_num_threads(1)
experiment = run_tiny_experiment(pretrain_steps=2,sft_steps=4)
prompt = format_prompt(TRAIN_EXAMPLES[0].instruction)
before = InferenceSession(experiment.tuned,experiment.tokenizer).request(prompt)
with tempfile.TemporaryDirectory(prefix='aibook-deploy-') as directory:
    path = Path(directory)/'مدل تمرین.pt'
    save_instruction_bundle(path,experiment.tuned,experiment.tokenizer)
    restored,restored_tokenizer = load_instruction_bundle(path)
after = InferenceSession(restored,restored_tokenizer,RequestLimits(128,8)).request(prompt)
assert before == after
print('same local request before/after loading:',after)
print('The temporary bundle is already removed. This is not a public service.')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">request_fits(prompt_tokens, new_tokens, context_limit, output_limit)</code> یک <bdi dir="ltr">bool</bdi> بدهد: <bdi dir="ltr">Prompt</bdi> حداقل یک <bdi dir="ltr">Token</bdi> داشته باشد، خروجی بین ۱ و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">output_limit</code> باشد و مجموعشان از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">context_limit</code> بیشتر نشود. ورودی‌های تمرین عدد صحیح‌اند؛ کاراکتر و <bdi dir="ltr">Token</bdi> را با هم مخلوط نکنید.</p>
</div>

In [ ]:
def request_fits(prompt_tokens, new_tokens, context_limit, output_limit):
    # TODO: خروجی از قبل جا رزرو می‌کند
    return None

In [ ]:
def test_exercise():
    result = request_fits(56,8,64,8)
    if result is None:
        return False
    assert result is True
    assert request_fits(57,8,64,8) is False
    assert request_fits(1,9,64,8) is False
    assert request_fits(0,1,64,8) is False
    assert request_fits(4,0,64,8) is False
    assert request_fits(4,-1,64,8) is False
    assert request_fits(1,1,2,1) is True
    length = len(restored_tokenizer.encode(prompt))
    assert request_fits(length,1,restored.config.context_length,8) is True
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: request_fits')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط بودجهٔ خروجی در همان درخواست را بیشتر کنید. برای ردشدن درخواست، فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ValueError</code> مورد انتظار را بگیرید؛ هر خروجی متناظر با همین مدل کم‌آموزش است و ادعای کیفیت ندارد.</p>
</div>

In [ ]:
session = InferenceSession(restored,restored_tokenizer,RequestLimits(128,8))
for count in (1,4,9):
    try:
        print(count,session.request(prompt,new_tokens=count))
    except ValueError as error:
        print(count,'expected request rejection:',error)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب <bdi dir="ltr">Vocabulary</bdi>ها را به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">set</code> تبدیل می‌کند؛ با این کار جابه‌جایی <bdi dir="ltr">ID</bdi>ها پنهان می‌شود. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">compatible_vocabulary(left_tokens,right_tokens)</code> فقط برای فهرست‌های هم‌ترتیب <bdi dir="ltr">True</bdi> بدهد. شباهت مجموعهٔ کاراکترها کافی نیست.</p>
</div>

In [ ]:
left_tokens = ['<|unk|>','ا','ب']
right_tokens = ['<|unk|>','ب','ا']
print('wrong unordered compatibility:',set(left_tokens)==set(right_tokens))
print('ID 1 now means:',left_tokens[1],right_tokens[1])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def compatible_vocabulary(left_tokens, right_tokens):
    # TODO: ترتیب، بخشی از قرارداد وزن است
    return None

In [ ]:
def test_repair():
    result = compatible_vocabulary(left_tokens,right_tokens)
    if result is None:
        return False
    assert result is False
    assert compatible_vocabulary(left_tokens,list(left_tokens)) is True
    assert compatible_vocabulary(left_tokens,left_tokens+['پ']) is False
    assert compatible_vocabulary(experiment.tokenizer.id_to_token,restored_tokenizer.id_to_token) is True
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: compatible_vocabulary')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">ذخیره، بارگذاری و تولید واقعاً اجرا شدند. بستهٔ این درس <bdi dir="ltr">Optimizer</bdi> و <bdi dir="ltr">RNG</bdi> لازم برای <bdi dir="ltr">Resume</bdi> را ندارد. <bdi dir="ltr">Layer</bdi> درخواست فقط محلی، همگام و محدود به تعداد <bdi dir="ltr">Token</bdi> است؛ <bdi dir="ltr">Timeout</bdi>، احراز هویت یا ایمنی سرویس عمومی را پیاده نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">آزمون <bdi dir="ltr">Round-trip</bdi> چه چیزی را ثابت کرد و برای اینکه یک سرویس عمومی قابل اتکا باشد چه قراردادهای دیگری هنوز لازم‌اند؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-15/chapter-03/83-deployment.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/83-deployment.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>